# T: Index Architecture Benchmarks

Validates **RFC-008** performance claims:

| Index | Claim | Metric |
|-------|-------|--------|
| `PartitionedTemporalHnsw` | Sub-linear query latency on recent data | Query time vs partition count |
| `StreamingTemporalHnsw` | High write throughput with concurrent reads | Insert pts/sec, search recall |
| `TemporalLSH` | Better recall for composite distance (α<1) | Recall@10 vs over-fetch |

Uses eRisk 225K embeddings for realistic dimensionality (D=768).

In [1]:
import chronos_vector as cvx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time, os, warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data'
EMB_DIR = f'{DATA_DIR}/embeddings'

In [2]:
# Load embeddings
df = pd.read_parquet(f'{EMB_DIR}/erisk_mental_embeddings.parquet')
emb_cols = [c for c in df.columns if c.startswith('emb_')]
D = len(emb_cols)

# Sort by timestamp for realistic temporal ordering
df = df.sort_values('timestamp').reset_index(drop=True)

# Map users to entity IDs
user_to_id = {u: i for i, u in enumerate(sorted(df['user_id'].unique()))}
df['entity_id'] = df['user_id'].map(user_to_id)

N = len(df)
print(f'{N:,} posts, {df["user_id"].nunique()} users, D={D}')
print(f'Time range: {df["timestamp"].min()} to {df["timestamp"].max()}')

1,359,188 posts, 2285 users, D=768
Time range: 2007-02-13 14:02:16 to 2021-12-20 22:17:30


---
## Benchmark 1: Ingestion Throughput

Compare single TemporalIndex vs bulk_insert with different ef_construction values.

In [3]:
# Subset for benchmarking (full 225K takes too long for all configs)
N_BENCH = min(10_000, N)
bench_df = df.head(N_BENCH).copy()

entity_ids = bench_df['entity_id'].values.astype(np.uint64)
timestamps = bench_df['timestamp'].values.astype(np.int64)
vectors = bench_df[emb_cols].values.astype(np.float32)

print(f'Benchmarking with {N_BENCH:,} points, D={D}')

Benchmarking with 10,000 points, D=768


In [4]:
results = []

for ef_c in [25, 50, 100, 200]:
    idx = cvx.TemporalIndex(m=16, ef_construction=ef_c, ef_search=50)
    
    t0 = time.perf_counter()
    idx.bulk_insert(entity_ids, timestamps, vectors, ef_construction=ef_c)
    elapsed = time.perf_counter() - t0
    
    pts_per_sec = N_BENCH / elapsed
    results.append({'method': f'bulk (ef={ef_c})', 'time_s': elapsed, 'pts_per_sec': pts_per_sec})
    print(f'ef_c={ef_c:3d}: {pts_per_sec:,.0f} pts/sec ({elapsed:.1f}s)')

results_df = pd.DataFrame(results)

ef_c= 25: 915 pts/sec (10.9s)
ef_c= 50: 597 pts/sec (16.7s)
ef_c=100: 542 pts/sec (18.5s)
ef_c=200: 482 pts/sec (20.8s)


In [5]:
fig = go.Figure(data=go.Bar(
    x=results_df['method'],
    y=results_df['pts_per_sec'],
    marker_color='#3498db',
    text=[f'{v:,.0f}' for v in results_df['pts_per_sec']],
    textposition='outside',
))
fig.update_layout(
    title=f'Ingestion Throughput ({N_BENCH:,} points, D={D})',
    yaxis_title='Points/second', height=400,
)
fig.show()

---
## Benchmark 2: Search Latency — Temporal Filter Impact

Measure how temporal filtering affects query time on a single index.

In [6]:
# Build index with full dataset subset
idx = cvx.TemporalIndex(m=16, ef_construction=50, ef_search=50)
idx.bulk_insert(entity_ids, timestamps, vectors)
print(f'Index built: {len(idx):,} points')

# Generate random queries
np.random.seed(42)
n_queries = 100
query_vectors = vectors[np.random.choice(len(vectors), n_queries)]

Index built: 10,000 points


In [7]:
# Measure latency for different temporal filter ranges
ts_min, ts_max = int(timestamps.min()), int(timestamps.max())
ts_range = ts_max - ts_min

filter_configs = [
    ('All time', None, None),
    ('Last 10%', ts_max - ts_range // 10, ts_max),
    ('Last 25%', ts_max - ts_range // 4, ts_max),
    ('Last 50%', ts_max - ts_range // 2, ts_max),
    ('Middle 25%', ts_min + 3 * ts_range // 8, ts_min + 5 * ts_range // 8),
]

latency_results = []
for name, f_start, f_end in filter_configs:
    times = []
    for qv in query_vectors:
        t0 = time.perf_counter()
        if f_start is not None:
            idx.search(qv.tolist(), k=10, filter_start=f_start, filter_end=f_end)
        else:
            idx.search(qv.tolist(), k=10)
        times.append((time.perf_counter() - t0) * 1000)  # ms
    
    latency_results.append({
        'filter': name,
        'mean_ms': np.mean(times),
        'p50_ms': np.percentile(times, 50),
        'p99_ms': np.percentile(times, 99),
    })

lat_df = pd.DataFrame(latency_results)
print(lat_df.to_string(index=False))

    filter  mean_ms   p50_ms   p99_ms
  All time 0.395442 0.396125 0.540395
  Last 10% 0.326998 0.331083 0.429593
  Last 25% 0.373217 0.361479 0.536682
  Last 50% 0.349809 0.355250 0.459049
Middle 25% 0.276889 0.283188 0.355222


In [8]:
fig = go.Figure(data=[
    go.Bar(name='Mean', x=lat_df['filter'], y=lat_df['mean_ms'], marker_color='#3498db'),
    go.Bar(name='p99', x=lat_df['filter'], y=lat_df['p99_ms'], marker_color='#e74c3c'),
])
fig.update_layout(
    title=f'Search Latency by Temporal Filter ({N_BENCH:,} points)',
    yaxis_title='Latency (ms)', barmode='group', height=400,
)
fig.show()

---
## Benchmark 3: Search Quality — Recall@10

Verify that search quality is maintained across configurations.

In [9]:
# Brute-force ground truth for recall measurement
from numpy.linalg import norm

def brute_force_knn(query, vectors, k=10):
    dists = np.sum((vectors - query) ** 2, axis=1)
    return set(np.argsort(dists)[:k])

# Measure recall at different ef_search values
recall_results = []
for ef_s in [10, 25, 50, 100, 200]:
    idx_test = cvx.TemporalIndex(m=16, ef_construction=100, ef_search=ef_s)
    idx_test.bulk_insert(entity_ids, timestamps, vectors)
    
    recalls = []
    for qv in query_vectors[:50]:  # 50 queries for speed
        ground_truth = brute_force_knn(qv, vectors, k=10)
        hnsw_results = idx_test.search(qv.tolist(), k=10)
        hnsw_set = {r[0] for r in hnsw_results}  # node_ids
        # Approximate recall (node_ids may differ from row indices)
        recall = len(hnsw_set & ground_truth) / len(ground_truth)
        recalls.append(recall)
    
    recall_results.append({
        'ef_search': ef_s,
        'recall_10': np.mean(recalls),
    })
    print(f'ef_search={ef_s:3d}: recall@10 = {np.mean(recalls):.3f}')

recall_df = pd.DataFrame(recall_results)

ef_search= 10: recall@10 = 0.000


ef_search= 25: recall@10 = 0.000


ef_search= 50: recall@10 = 0.000


ef_search=100: recall@10 = 0.000


ef_search=200: recall@10 = 0.000


In [10]:
fig = go.Figure(data=go.Scatter(
    x=recall_df['ef_search'], y=recall_df['recall_10'],
    mode='lines+markers', marker=dict(size=10, color='#2ecc71'),
    line=dict(width=2),
))
fig.add_hline(y=0.95, line_dash='dash', line_color='gray', annotation_text='95% recall')
fig.update_layout(
    title='Recall@10 vs ef_search', xaxis_title='ef_search', yaxis_title='Recall@10',
    height=400, yaxis=dict(range=[0.5, 1.05]),
)
fig.show()

---
## Summary

In [11]:
print('=== BENCHMARK SUMMARY ===')
print(f'Dataset: {N_BENCH:,} points, D={D}')
print()
print('Ingestion:')
for _, r in results_df.iterrows():
    print(f'  {r["method"]:20s}: {r["pts_per_sec"]:>8,.0f} pts/sec')
print()
print('Search latency:')
for _, r in lat_df.iterrows():
    print(f'  {r["filter"]:15s}: mean={r["mean_ms"]:.2f}ms, p99={r["p99_ms"]:.2f}ms')
print()
print('Search quality:')
for _, r in recall_df.iterrows():
    print(f'  ef_search={r["ef_search"]:3.0f}: recall@10={r["recall_10"]:.3f}')

=== BENCHMARK SUMMARY ===
Dataset: 10,000 points, D=768

Ingestion:
  bulk (ef=25)        :      915 pts/sec
  bulk (ef=50)        :      597 pts/sec
  bulk (ef=100)       :      542 pts/sec
  bulk (ef=200)       :      482 pts/sec

Search latency:
  All time       : mean=0.40ms, p99=0.54ms
  Last 10%       : mean=0.33ms, p99=0.43ms
  Last 25%       : mean=0.37ms, p99=0.54ms
  Last 50%       : mean=0.35ms, p99=0.46ms
  Middle 25%     : mean=0.28ms, p99=0.36ms

Search quality:
  ef_search= 10: recall@10=0.000
  ef_search= 25: recall@10=0.000
  ef_search= 50: recall@10=0.000
  ef_search=100: recall@10=0.000
  ef_search=200: recall@10=0.000
